# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Read Bronze Table

In [0]:
df = spark.table("workspace.bronze.erp_px_cat_g1v2_raw")
df.display()

### Trimming

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))

### Normalize Maintenance Flag to Boolean

In [0]:
df = df.withColumn(
  "MAINTENANCE", 
  (
    F.when(F.upper(col("MAINTENANCE")) == "YES", "True")
     .when(F.upper(col("MAINTENANCE")) == "NO", "False")
     .otherwise(col("MAINTENANCE"))
  ).cast("boolean")
)

### Renaming Columns

In [0]:
RENAME_MAP = {
    "ID": "category_id",
    "CAT": "category_name",
    "SUBCAT": "subcategory_name",
    "MAINTENANCE": "maintenance_flag"
}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

# Check Dataframe

In [0]:
df.display()

# Save Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_product_category")

# Check Silver Table

In [0]:
%sql
SELECT *
FROM workspace.silver.erp_product_category